In [ ]:
!pip install numpy pandas matplotlib seaborn scikit-learn tensorflow

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_curve, auc
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow as tf

In [ ]:
from google.colab import files
import pandas as pd
uploaded = files.upload()
df = pd.read_csv("PhishFOE_ds.csv", header=0)

Saving PhishFOE_ds.csv to PhishFOE_ds.csv


In [ ]:
if 'URL' in df.columns:
    df.drop(columns=['URL'], inplace=True)

# Supprimer les colonnes non numériques
df = df.select_dtypes(include=[np.number, 'int64', 'float64'])

# Vérifier s’il reste bien la colonne cible
if 'label' not in df.columns:
    raise ValueError("La colonne 'label' est absente après nettoyage. Vérifiez le nom ou le contenu du dataset.")

# Séparation features/target
X = df.drop(columns=['label']).values
y = df['label'].values

# --- RFECV Feature Selection ---
estimator = LogisticRegression(solver='liblinear', max_iter=1000)
rfecv = RFECV(estimator, step=1, cv=5, scoring='accuracy', n_jobs=-1)
rfecv.fit(X, y)

print(f"Optimal number of features: {rfecv.n_features_}")
print(f"Selected features mask: {rfecv.support_}")

# Keep only selected features
X = rfecv.transform(X)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Reshape for LSTM
X = X.reshape(X.shape[0], 1, X.shape[1])

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- Define LSTM Model ---
def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.3),

        LSTM(128, return_sequences=False),
        BatchNormalization(),
        Dropout(0.4),

        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),

        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    return model

# Build model
model = build_lstm((X_train.shape[1], X_train.shape[2]))
model.summary()

# Callbacks
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ModelCheckpoint('best_lstm_model.h5', save_best_only=True)
]

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    callbacks=callbacks,
    class_weight={0: 1, 1: 1.2}
)

# Evaluation
y_pred = (model.predict(X_test) > 0.5).astype(int)

# Print classification report with 4-digit precision
print(classification_report(y_test, y_pred, digits=4))

Optimal number of features: 24
Selected features mask: [ True  True  True  True  True  True  True  True  True  True False  True
  True  True  True  True  True  True  True  True  True  True False  True
 False False  True  True False]


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 1, 64)          │        22,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 1, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 156,673 (612.00 KB)

...

1011/1011 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.9722 - loss: 0.0850 - precision_1: 0.9730 - recall_1: 0.9699 - val_accuracy: 0.9748 - val_loss: 0.0699 - val_precision_1: 0.9797 - val_recall_1: 0.9685
Epoch 27/100
1009/1011 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9725 - loss: 0.0852 - precision_1: 0.9739 - recall_1: 0.9695

1011/1011 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9725 - loss: 0.0852 - precision_1: 0.9739 - recall_1: 0.9695 - val_accuracy: 0.9743 - val_loss: 0.0698 - val_precision_1: 0.9738 - val_recall_1: 0.9737
Epoch 28/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9725 - loss: 0.0838 - precision_1: 0.9732 - recall_1: 0.9706 - val_accuracy: 0.9738 - val_loss: 0.0704 - val_precision_1: 0.9719 - val_recall_1: 0.9746
Epoch 29/100
1010/1011 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9720 - loss: 0.0849 - precision_1: 0.9737 - recall_1: 0.9687

1011/1011 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - accuracy: 0.9720 - loss: 0.0849 - precision_1: 0.9737 - recall_1: 0.9687 - val_accuracy: 0.9746 - val_loss: 0.0697 - val_precision_1: 0.9760 - val_recall_1: 0.9721
Epoch 30/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.9732 - loss: 0.0834 - precision_1: 0.9753 - recall_1: 0.9701 - val_accuracy: 0.9738 - val_loss: 0.0713 - val_precision_1: 0.9721 - val_recall_1: 0.9745
Epoch 31/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9718 - loss: 0.0847 - precision_1: 0.9738 - recall_1: 0.9688 - val_accuracy: 0.9730 - val_loss: 0.0704 - val_precision_1: 0.9690 - val_recall_1: 0.9760
Epoch 32/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.9735 - loss: 0.0817 - precision_1: 0.9748 - recall_1: 0.9710 - val_accuracy: 0.9720 - val_loss: 0.0727 - val_precision_1: 0.9701 - val_recall_1: 0.9728
Epoch 33/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.9734 - loss: 0.0796 - precision_1: 0.9750 - r

1011/1011 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.9720 - loss: 0.0858 - precision_1: 0.9735 - recall_1: 0.9698 - val_accuracy: 0.9748 - val_loss: 0.0676 - val_precision_1: 0.9777 - val_recall_1: 0.9706
Epoch 36/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.9737 - loss: 0.0819 - precision_1: 0.9747 - recall_1: 0.9717 - val_accuracy: 0.9743 - val_loss: 0.0686 - val_precision_1: 0.9778 - val_recall_1: 0.9696
Epoch 37/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9713 - loss: 0.0833 - precision_1: 0.9730 - recall_1: 0.9685 - val_accuracy: 0.9740 - val_loss: 0.0702 - val_precision_1: 0.9704 - val_recall_1: 0.9766
Epoch 38/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.9743 - loss: 0.0797 - precision_1: 0.9764 - recall_1: 0.9711 - val_accuracy: 0.9749 - val_loss: 0.0684 - val_precision_1: 0.9742 - val_recall_1: 0.9745
Epoch 39/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.9746 - loss: 0.0799 - precision_1: 0.9765 - r

1011/1011 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.9737 - loss: 0.0829 - precision_1: 0.9767 - recall_1: 0.9697 - val_accuracy: 0.9749 - val_loss: 0.0669 - val_precision_1: 0.9746 - val_recall_1: 0.9741
Epoch 45/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9751 - loss: 0.0785 - precision_1: 0.9765 - recall_1: 0.9724 - val_accuracy: 0.9745 - val_loss: 0.0685 - val_precision_1: 0.9710 - val_recall_1: 0.9771
Epoch 46/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9744 - loss: 0.0771 - precision_1: 0.9754 - recall_1: 0.9721 - val_accuracy: 0.9741 - val_loss: 0.0695 - val_precision_1: 0.9685 - val_recall_1: 0.9790
Epoch 47/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.9747 - loss: 0.0800 - precision_1: 0.9749 - recall_1: 0.9731 - val_accuracy: 0.9745 - val_loss: 0.0698 - val_precision_1: 0.9703 - val_recall_1: 0.9778
Epoch 48/100
1011/1011 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - accuracy: 0.9740 - loss: 0.0812 - precision_1: 0.9756 - r

In [ ]:
# Normalisation des données
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
# Redimensionnement pour LSTM (samples, timesteps, features)
X = X.reshape(X.shape[0], 1, X.shape[1])

In [ ]:
# Split des données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# Définition du modèle LSTM
def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.3),

        LSTM(128, return_sequences=False),
        BatchNormalization(),
        Dropout(0.4),

        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),

        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model

In [ ]:
model = build_lstm((X_train.shape[1], X_train.shape[2]))
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 1, 64)               │          38,912 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 1, 64)               │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1, 64)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 128)                 │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             257 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 172,801 (675.00 KB)

 Trainable params: 171,905 (671.50 KB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
# Construction et entraînement du modèle
model = build_lstm((X_train.shape[1], X_train.shape[2]))

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ModelCheckpoint('best_lstm_model.h5', save_best_only=True)
]

history = model.fit(X_train, y_train, epochs=100, batch_size=64, validation_split=0.2, callbacks=callbacks, class_weight={0: 1, 1: 1.2})

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7916 - loss: 0.5309 - precision: 0.7798 - recall: 0.8035 

115/115 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.7922 - loss: 0.5295 - precision: 0.7805 - recall: 0.8041 - val_accuracy: 0.8907 - val_loss: 0.5521 - val_precision: 0.8298 - val_recall: 0.9848
Epoch 2/100
111/115 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9253 - loss: 0.2205 - precision: 0.9100 - recall: 0.9430

115/115 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9254 - loss: 0.2202 - precision: 0.9103 - recall: 0.9427 - val_accuracy: 0.9508 - val_loss: 0.3409 - val_precision: 0.9444 - val_recall: 0.9587
Epoch 3/100
113/115 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9295 - loss: 0.2014 - precision: 0.9204 - recall: 0.9406

115/115 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9295 - loss: 0.2013 - precision: 0.9204 - recall: 0.9406 - val_accuracy: 0.9568 - val_loss: 0.1818 - val_precision: 0.9647 - val_recall: 0.9490
Epoch 4/100
113/115 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9393 - loss: 0.1904 - precision: 0.9411 - recall: 0.9363

115/115 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9393 - loss: 0.1901 - precision: 0.9410 - recall: 0.9365 - val_accuracy: 0.9579 - val_loss: 0.1241 - val_precision: 0.9587 - val_recall: 0.9577
Epoch 5/100
113/115 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9429 - loss: 0.1590 - precision: 0.9381 - recall: 0.9487

115/115 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9429 - loss: 0.1590 - precision: 0.9382 - recall: 0.9488 - val_accuracy: 0.9623 - val_loss: 0.1207 - val_precision: 0.9671 - val_recall: 0.9577
Epoch 6/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9567 - loss: 0.1336 - precision: 0.9493 - recall: 0.9650 - val_accuracy: 0.9590 - val_loss: 0.1232 - val_precision: 0.9578 - val_recall: 0.9609
Epoch 7/100
111/115 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9534 - loss: 0.1467 - precision: 0.9458 - recall: 0.9623

115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9533 - loss: 0.1466 - precision: 0.9458 - recall: 0.9621 - val_accuracy: 0.9617 - val_loss: 0.1195 - val_precision: 0.9650 - val_recall: 0.9587
Epoch 8/100
113/115 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9563 - loss: 0.1307 - precision: 0.9516 - recall: 0.9607

115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9562 - loss: 0.1308 - precision: 0.9516 - recall: 0.9606 - val_accuracy: 0.9634 - val_loss: 0.1121 - val_precision: 0.9662 - val_recall: 0.9609
Epoch 9/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9514 - loss: 0.1408 - precision: 0.9456 - recall: 0.9588 - val_accuracy: 0.9617 - val_loss: 0.1143 - val_precision: 0.9630 - val_recall: 0.9609
Epoch 10/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9593 - loss: 0.1200 - precision: 0.9556 - recall: 0.9627 - val_accuracy: 0.9634 - val_loss: 0.1123 - val_precision: 0.9672 - val_recall: 0.9598
Epoch 11/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9595 - loss: 0.1234 - precision: 0.9503 - recall: 0.9684 - val_accuracy: 0.9612 - val_loss: 0.1131 - val_precision: 0.9712 - val_recall: 0.9511
Epoch 12/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9604 - loss: 0.1108 - precision: 0.9576 - recall: 0.9617 - val_accuracy: 0.9634 - val_loss: 0

In [ ]:
# Évaluation du modéle
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, y_pred))

72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      1143
           1       0.95      0.95      0.95      1143

    accuracy                           0.95      2286
   macro avg       0.95      0.95      0.95      2286
weighted avg       0.95      0.95      0.95      2286

